# PathSnap private GPU experiments
Fresh raw preprocessing, five models, grouped CV and TPE. No GitHub access.

In [ ]:
from pathlib import Path
import os,sys,subprocess,zipfile,json,shutil,time
ROOT=Path('/kaggle/working/PathSnap_complete'); ROOT.mkdir(exist_ok=True)
archives=list(Path('/kaggle/input').rglob('pathsnap_source.zip'))
if archives:
 with zipfile.ZipFile(archives[0]) as z: z.extractall(ROOT)
else:
 # Kaggle expands ZIP dataset uploads on ingestion.
 markers=list(Path('/kaggle/input').rglob('train_complete.py'))
 assert len(markers)==1,[(str(x)) for x in Path('/kaggle/input').rglob('*')][:80]
 source=markers[0].parents[1]
 shutil.copytree(source,ROOT,dirs_exist_ok=True)

# Dataset ingestion recursively expands ZIPs; reconstruct the exact raw layout.
raw_zip=ROOT/'data/raw/Student folder.zip'
if not raw_zip.exists():
 raw_dir=ROOT/'data/raw/Student folder'
 assert raw_dir.exists(),str(raw_dir)
 with zipfile.ZipFile(raw_zip,'w',zipfile.ZIP_DEFLATED) as z:
  for p in raw_dir.rglob('*'):
   if p.is_file(): z.write(p,str(p.relative_to(raw_dir)))
os.chdir(ROOT)
(ROOT/'scripts/train_complete.py').write_text('"""Fresh five-family experiments, grouped CV and genuine sequential TPE search."""\nimport os,sys,json,time,platform,subprocess,hashlib\nfrom pathlib import Path\nROOT=Path(__file__).resolve().parents[1]; sys.path.insert(0,str(ROOT))\nimport numpy as np\nimport pandas as pd\nimport joblib,optuna\nfrom sklearn.model_selection import GroupKFold\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.neighbors import KNeighborsClassifier\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,log_loss,confusion_matrix\nfrom threadpoolctl import threadpool_limits\nfrom pathsnap.pipeline import FEATURES,KEY,read_trip,features,score\nSEED=261118\nOUT=ROOT/\'results\'; OUT.mkdir(exist_ok=True)\nBASE={\n \'logistic\':{\'C\':1.0},\n \'knn\':{\'n_neighbors\':25,\'max_fit_rows\':160000},\n \'forest\':{\'n_estimators\':80,\'max_depth\':16,\'min_samples_leaf\':10},\n \'xgboost\':{\'n_estimators\':240,\'max_depth\':5,\'learning_rate\':0.07,\'min_child_weight\':12,\'subsample\':0.85,\'colsample_bytree\':0.9,\'reg_lambda\':5.0},\n \'mlp\':{\'epochs\':12,\'learning_rate\':0.001,\'weight_decay\':0.0001,\'batch_size\':16384}}\n\ndef write(name,obj): (OUT/name).write_text(json.dumps(obj,indent=2,default=str)+\'\\n\')\n\ndef load_data(split):\n c=pd.read_parquet(ROOT/f\'data/processed/candidates_{split}.parquet\')\n labels=pd.read_parquet(ROOT/f\'data/processed/reference_labels_{split}.parquet\')\n chunks=[]; xx=[]\n for trip,rows in c.groupby(\'trip_id\',sort=True):\n  t,n,a=read_trip(ROOT/\'data/raw/Student folder.zip\',split,trip)\n  chunks.append(rows); xx.append(features(t,rows,n,a))\n c=pd.concat(chunks,ignore_index=True)\n x=np.concatenate(xx)\n l=labels.merge(c[KEY[:-1]].drop_duplicates().assign(any_candidate=True),on=KEY[:-1],how=\'left\',validate=\'one_to_one\')\n c=c.merge(labels[KEY[:-1]+[\'reference_arc_id\',\'alignment_status\',\'reference_in_candidates\',\'usable_for_training\']],on=KEY[:-1],validate=\'many_to_one\')\n y=c.arc_id.eq(c.reference_arc_id).fillna(False).to_numpy(dtype=\'uint8\')\n usable=(c.alignment_status.eq(\'aligned\')&c.reference_in_candidates).to_numpy()\n return dict(c=c,x=x,y=y,usable=usable,labels=l)\n\ndef take(data,trips):\n m=data[\'c\'].trip_id.isin(trips).to_numpy()\n return dict(c=data[\'c\'].loc[m].reset_index(drop=True),x=data[\'x\'][m],y=data[\'y\'][m],usable=data[\'usable\'][m],labels=data[\'labels\'].loc[data[\'labels\'].trip_id.isin(trips)])\n\ndef metrics(data,p):\n c=data[\'c\']; m=data[\'usable\']; y=data[\'y\'][m]; pp=p[m]; pred=pp>=0.5\n w=c[KEY+[\'candidate_rank\']].assign(score=p).sort_values([\'score\',\'candidate_rank\',\'arc_id\'],ascending=[False,True,True]).drop_duplicates(KEY[:-1])\n aligned=data[\'labels\'].loc[data[\'labels\'].alignment_status.eq(\'aligned\')]\n z=aligned.merge(w[KEY],on=KEY[:-1],how=\'left\',validate=\'one_to_one\')\n correct=z.arc_id.eq(z.reference_arc_id).fillna(False)\n trip_acc=correct.groupby(z.trip_id).mean()\n cm=confusion_matrix(y,pred,labels=[0,1]).ravel()\n return dict(top1_accuracy=float(correct.mean()),trip_macro_top1=float(trip_acc.mean()),macro_f1=float(f1_score(y,pred,average=\'macro\',zero_division=0)),\n  precision=float(precision_score(y,pred,zero_division=0)),recall=float(recall_score(y,pred,zero_division=0)),positive_f1=float(f1_score(y,pred,zero_division=0)),\n  candidate_accuracy=float(accuracy_score(y,pred)),log_loss=float(log_loss(y,pp,labels=[0,1])),aligned=len(z),\n  candidate_misses=int((~z.reference_in_candidates).sum()),candidate_miss_rate=float((~z.reference_in_candidates).mean()),\n  ambiguous_excluded=int(data[\'labels\'].alignment_status.eq(\'ambiguous\').sum()),unmatched_excluded=int(data[\'labels\'].reference_arc_id.isna().sum()),\n  tn=int(cm[0]),fp=int(cm[1]),fn=int(cm[2]),tp=int(cm[3]))\n\ndef fit(name,params,data,tag,validation=None):\n import torch\n np.random.seed(SEED); torch.manual_seed(SEED)\n mask=data[\'usable\']; x=data[\'x\'][mask]; y=data[\'y\'][mask]\n start=time.perf_counter(); history=[]\n artifact=dict(name=name,params=params,features=FEATURES,kind=\'sklearn\',seed=SEED)\n if name==\'logistic\': estimator=make_pipeline(StandardScaler(),LogisticRegression(C=params[\'C\'],max_iter=300,random_state=SEED))\n elif name==\'knn\':\n  if len(x)>params[\'max_fit_rows\']:\n   indices=np.random.default_rng(SEED).choice(len(x),params[\'max_fit_rows\'],replace=False); x=x[indices]; y=y[indices]\n  estimator=make_pipeline(StandardScaler(),KNeighborsClassifier(n_neighbors=params[\'n_neighbors\'],weights=\'distance\',algorithm=\'kd_tree\',n_jobs=4))\n elif name==\'forest\': estimator=RandomForestClassifier(**params,n_jobs=4,random_state=SEED,max_features=\'sqrt\')\n elif name==\'xgboost\':\n  from xgboost import XGBClassifier\n  estimator=XGBClassifier(**params,device=\'cuda\' if torch.cuda.is_available() else \'cpu\',tree_method=\'hist\',objective=\'binary:logistic\',eval_metric=\'logloss\',n_jobs=4,random_state=SEED)\n elif name==\'mlp\':\n  scaler=StandardScaler().fit(x); x=scaler.transform(x).astype(\'float32\')\n  device=\'cuda\' if torch.cuda.is_available() else \'cpu\'\n  torch.set_num_threads(4)\n  net=torch.nn.Sequential(torch.nn.Linear(len(FEATURES),64),torch.nn.ReLU(),torch.nn.Linear(64,32),torch.nn.ReLU(),torch.nn.Linear(32,1)).to(device)\n  opt=torch.optim.AdamW(net.parameters(),lr=params[\'learning_rate\'],weight_decay=params[\'weight_decay\'])\n  xx=torch.from_numpy(x).to(device); yy=torch.from_numpy(y.astype(\'float32\')).to(device)\n  if validation is not None:\n   vm=validation[\'usable\']; vx=torch.from_numpy(scaler.transform(validation[\'x\'][vm]).astype(\'float32\')).to(device)\n   vy=torch.from_numpy(validation[\'y\'][vm].astype(\'float32\')).to(device)\n  for epoch in range(params[\'epochs\']):\n   net.train(); order=torch.randperm(len(x),device=device); total=0\n   for idx in order.split(params[\'batch_size\']):\n    opt.zero_grad(set_to_none=True); loss=torch.nn.functional.binary_cross_entropy_with_logits(net(xx[idx]).ravel(),yy[idx]); loss.backward(); opt.step(); total+=loss.item()*len(idx)\n   row=dict(run=tag,epoch=epoch+1,train_bce=total/len(x))\n   if validation is not None:\n    net.eval()\n    with torch.no_grad(): row[\'validation_bce\']=float(torch.nn.functional.binary_cross_entropy_with_logits(net(vx).ravel(),vy).item())\n   history.append(row)\n  artifact.update(kind=\'torch\',state={k:v.detach().cpu() for k,v in net.state_dict().items()},scaler=scaler,device_trained=device)\n else: raise ValueError(name)\n if name!=\'mlp\': estimator.fit(x,y); artifact[\'estimator\']=estimator\n artifact[\'fit_rows\']=len(x); artifact[\'fit_seconds\']=time.perf_counter()-start\n return artifact,history\n\ndef evaluate(model,data,name,phase,fold=None):\n start=time.perf_counter(); p=score(model,data[\'x\']); elapsed=time.perf_counter()-start\n return dict(model=name,phase=phase,fold=fold,inference_seconds=elapsed,fit_seconds=model[\'fit_seconds\'],fit_rows=model[\'fit_rows\'],**metrics(data,p)),p\n\ndef main():\n import torch\n if os.environ.get(\'PATHSNAP_REQUIRE_GPU\')==\'1\' and not torch.cuda.is_available(): raise RuntimeError(\'GPU run requested but CUDA is unavailable\')\n gpu=subprocess.run([\'nvidia-smi\'],capture_output=True,text=True) if torch.cuda.is_available() else None\n (OUT/\'nvidia-smi.txt\').write_text(gpu.stdout if gpu else \'No CUDA GPU\')\n write(\'environment.json\',dict(python=sys.version,platform=platform.platform(),torch=torch.__version__,cuda=torch.version.cuda,gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,seed=SEED,started_utc=pd.Timestamp.now(tz=\'UTC\').isoformat()))\n subprocess.run([sys.executable,\'-m\',\'pip\',\'freeze\'],stdout=(OUT/\'environment_freeze.txt\').open(\'w\'),check=True)\n train=load_data(\'train\'); test=load_data(\'student_test\')\n trainids=sorted(train[\'c\'].trip_id.unique()); testids=sorted(test[\'c\'].trip_id.unique())\n assert len(trainids)==80 and len(testids)==10 and not set(trainids)&set(testids)\n joblib.dump({\'train\':train,\'student_test\':test},OUT/\'feature_cache.joblib\',compress=0)\n # Split complete trips, balancing observation counts through the label timeline.\n folds=[]; assignments=[]\n for k,(i,j) in enumerate(GroupKFold(3).split(train[\'labels\'],groups=train[\'labels\'].trip_id),1):\n  fitids=train[\'labels\'].iloc[i].trip_id.unique(); valid=train[\'labels\'].iloc[j].trip_id.unique()\n  assert not set(fitids)&set(valid); folds.append((fitids,valid))\n  assignments.extend(dict(trip_id=t,fold=k) for t in valid)\n pd.DataFrame(assignments).to_csv(OUT/\'cv_folds.csv\',index=False)\n write(\'protocol.json\',dict(seed=SEED,features=FEATURES,folds=3,selection_objective=\'mean grouped-CV observation top-1 accuracy; candidate misses count as errors\',binary_metrics=\'Aligned observations with reference inside candidates, threshold 0.5; binary candidate macro F1, not arc-class F1\',knn_sample=\'Uniform without replacement, at most 160000 candidate rows from each fit partition\',evaluator_data_read=False,configs=BASE))\n rows=[]; history=[]; model_cv={}\n for name,params in BASE.items():\n  cv=[]\n  for k,(fitids,valid) in enumerate(folds,1):\n   print(f\'START {name} fold {k}\',flush=True)\n   a,h=fit(name,params,take(train,fitids),f\'{name}_fold{k}\',take(train,valid) if name==\'mlp\' else None); history+=h\n   row,p=evaluate(a,take(train,valid),name,\'cv\',k); rows.append(row); cv.append(row[\'top1_accuracy\'])\n   print(json.dumps(row),flush=True); pd.DataFrame(rows).to_csv(OUT/\'model_metrics.csv\',index=False)\n  model_cv[name]=float(np.mean(cv))\n  a,h=fit(name,params,train,f\'{name}_full\'); history+=h; joblib.dump(a,OUT/f\'{name}.joblib\',compress=3)\n  for phase,data in [(\'train_fit\',train),(\'student_test\',test)]:\n   row,p=evaluate(a,data,name,phase); rows.append(row)\n   if phase==\'student_test\': data[\'c\'][KEY].assign(score=p).to_parquet(OUT/f\'{name}_scores.parquet\',index=False)\n   print(json.dumps(row),flush=True)\n  pd.DataFrame(rows).to_csv(OUT/\'model_metrics.csv\',index=False); pd.DataFrame(history).to_csv(OUT/\'learning_curves.csv\',index=False)\n # Boosting is a promising probability model and efficiently supports GPU TPE.\n # Final selection compares its tuned CV objective with every baseline.\n study=optuna.create_study(direction=\'maximize\',sampler=optuna.samplers.TPESampler(seed=SEED,n_startup_trials=4),storage=f\'sqlite:///{OUT}/search.sqlite3\',study_name=\'pathsnap_xgboost\',load_if_exists=True)\n study.enqueue_trial(BASE[\'xgboost\'])\n def objective(trial):\n  params=dict(n_estimators=trial.suggest_int(\'n_estimators\',120,480,step=60),max_depth=trial.suggest_int(\'max_depth\',3,8),learning_rate=trial.suggest_float(\'learning_rate\',0.02,0.2,log=True),min_child_weight=trial.suggest_int(\'min_child_weight\',4,64),subsample=trial.suggest_float(\'subsample\',0.65,1.0),colsample_bytree=trial.suggest_float(\'colsample_bytree\',0.65,1.0),reg_lambda=trial.suggest_float(\'reg_lambda\',0.5,30,log=True))\n  scores=[]; f1=[]\n  for k,(fi,vi) in enumerate(folds,1):\n   a,_=fit(\'xgboost\',params,take(train,fi),f\'trial{trial.number}_{k}\'); row,_=evaluate(a,take(train,vi),\'xgboost_tuned\',\'search\',k); scores.append(row[\'top1_accuracy\']); f1.append(row[\'macro_f1\'])\n  trial.set_user_attr(\'fold_top1\',scores); trial.set_user_attr(\'fold_macro_f1\',f1)\n  return float(np.mean(scores))\n study.optimize(objective,n_trials=12,callbacks=[lambda s,t:s.trials_dataframe().to_csv(OUT/\'search_trials.csv\',index=False)])\n best=study.best_trial; model_cv[\'xgboost_tuned\']=best.value\n write(\'tuning.json\',dict(best_trial=best.number,best_params=best.params,cv_top1=best.value,trials=len(study.trials),sampler=\'TPE\',startup_trials=4,search_space={\'n_estimators\':\'120..480 step 60\',\'max_depth\':\'3..8\',\'learning_rate\':\'log 0.02..0.2\',\'min_child_weight\':\'4..64\',\'subsample\':\'0.65..1\',\'colsample_bytree\':\'0.65..1\',\'reg_lambda\':\'log 0.5..30\'},selected_on=\'training trips only\'))\n a,h=fit(\'xgboost\',best.params,train,\'xgboost_tuned_full\'); joblib.dump(a,OUT/\'xgboost_tuned.joblib\',compress=3)\n for k in range(3):\n  rows.append(dict(model=\'xgboost_tuned\',phase=\'cv\',fold=k+1,top1_accuracy=best.user_attrs[\'fold_top1\'][k],macro_f1=best.user_attrs[\'fold_macro_f1\'][k]))\n for phase,data in [(\'train_fit\',train),(\'student_test\',test)]:\n  row,p=evaluate(a,data,\'xgboost_tuned\',phase); rows.append(row)\n  if phase==\'student_test\': data[\'c\'][KEY].assign(score=p).to_parquet(OUT/\'xgboost_tuned_scores.parquet\',index=False)\n selected=max(model_cv,key=model_cv.get)\n joblib.dump(joblib.load(OUT/f\'{selected}.joblib\'),OUT/\'final_model.joblib\',compress=3)\n write(\'selection.json\',dict(selected_model=selected,cv_top1=model_cv,reason=\'Highest mean three-fold held-out-trip top-1; student-test scores not used for selection\',optimism_note=\'Tuned CV is used both for search and selection, not an unbiased nested-CV estimate. Student-test remains the development comparison.\'))\n pd.DataFrame(rows).to_csv(OUT/\'model_metrics.csv\',index=False)\n print(\'SELECTED\',selected,model_cv,flush=True)\n\nif __name__==\'__main__\':\n with threadpool_limits(limits=4): main()\n')
subprocess.run([sys.executable,'-m','pip','install','-q','scikit-learn==1.7.2','optuna>=4,<5','folium>=0.20','pyarrow','nbclient','nbformat'],check=True)
import torch
assert torch.cuda.is_available(),'CUDA required'
print('GPU:',torch.cuda.get_device_name(0),flush=True)
os.environ['PATHSNAP_REQUIRE_GPU']='1'
logs=ROOT/'results'; logs.mkdir(exist_ok=True)
def run(label,args):
 print('START',label,flush=True)
 with (logs/(label+'.log')).open('w') as f:
  p=subprocess.Popen([sys.executable,*args],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
  for line in p.stdout: print(line,end='',flush=True); f.write(line); f.flush()
  assert p.wait()==0,label
# Rebuild the supplied raw archive, not prior experiments or saved predictions.
import nbformat
from nbclient import NotebookClient
nb=nbformat.read(ROOT/'notebooks/gps_preprocessing.ipynb',as_version=4)
NotebookClient(nb,timeout=None,kernel_name='python3',resources={'metadata':{'path':str(ROOT)}}).execute()
nbformat.write(nb,ROOT/'notebooks/gps_preprocessing_executed.ipynb')
print('RAW PREPROCESSING COMPLETE',flush=True)
run('candidates',['scripts/generate_candidates.py'])
run('reference_alignment',['scripts/generate_reference_labels.py'])
run('training',['scripts/train_complete.py'])
with zipfile.ZipFile('/kaggle/working/pathsnap_gpu_results.zip','w',zipfile.ZIP_DEFLATED) as z:
 for folder in ['results','data/processed','notebooks']:
  for p in (ROOT/folder).rglob('*'):
   if p.is_file() and p.name!='feature_cache.joblib': z.write(p,str(p.relative_to(ROOT)))
print('FINISHED: pathsnap_gpu_results.zip',flush=True)
